# **ÚLTIMA VERSIÓN -> Martes 26 de Mayo a las 15:44**

#**Dataset medicamentos PAMI**

Se busca identificar desde el dataset obtenido en Datos.gob.ar
- Qué medicamentos aparecen
- Qué porcentaje cubre PAMI;
- Qué precio figura en la base;

---

**NOTAS DE IMPORTANCIA SOBRE EL DATASET**

* Un análisis preliminar, comparando el precio de los medicamentos con los publicados en sitios webs donde se puede buscar el medicamento por marca, laboratorio, comprimido y compuesto corrobora que la base de datos se encuentra actualizada
* Se corrobora que el precio que figura en el dataset ya es con el descuento correspondiente aplicado

In [ ]:
# 1. IMPORTAR LIBRERÍAS
# =========================

import pandas as pd
import numpy as np
from google.colab import drive
drive.mount('/content/drive')

## **1. Carga de la base de medicamentos PAMI**

**Opción 1**
Usar la URL del dataset, permite tener la base actualizada automaticamente

http://datos.pami.org.ar/dataset/b40f7569-3a23-46bf-8a45-dd7cff41e725/resource/92ad6862-af8e-4047-b2cb-4bfef705feb3/download/afiliados20260504_100332.xlsx

**Opción 2**
En caso de que no corra del URL, lo que puede deberse a que la página no fluye (problema que sucedio en google colab) se sube el dataset descargado de dicho link

---
**NOTA**

El codigo va a ejecutar la opcion 1, pero si la web de pami no responde,  entonces va a ejecutar la opción 2, que basicamente es un archivo de respaldo dejado en el codigo.
Este archivo de respaldo se sube al codigo una vez que se descargo desde el link que google colab no pudo ejecutar, eso implica que trae exactamente el mismo data (columnas) pero sin asegurar que esta actualizado en tiempo real. Motivo por el cual se deja el mensaje donde esto se aclare

In [ ]:
# OPCIÓN 1 + y OPCION 2 (respaldo): CARGA DEL DATASET PAMI
# ==================================================

import pandas as pd

# 1. URL oficial del dataset de medicamentos PAMI
url_medicamentos = "http://datos.pami.org.ar/dataset/b40f7569-3a23-46bf-8a45-dd7cff41e725/resource/92ad6862-af8e-4047-b2cb-4bfef705feb3/download/afiliados20260504_100332.xlsx"

# 2. Archivo de respaldo guardado
  # En Colab, por ahora, este archivo debe estar subido a la sesión o montado desde Drive.
archivo_medicamentos_respaldo = "/content/drive/MyDrive/Licenciatura en ciencia política/En curso/Ciencia de datos para polítologos/Proyecto final/CODIGO_PROYECTO/CODIGO_MEDICAMENTOS/Dataset_Medicamentos_PAMI.xlsx"

try:
# 3. Intentamos cargar la base desde internet
    df_medicamentos = pd.read_excel(url_medicamentos)
    print("La base se cargó correctamente desde PAMI.\n")

# 4. Si falla la conexión, usamos la copia de respaldo del proyecto
except Exception as error:
    print("No se pudo cargar la base desde PAMI en este momento. \n Se usará la última versión de respaldo disponible en el proyecto, puede que los precios no esten actualizados. \n")

    df_medicamentos = pd.read_excel(archivo_medicamentos_respaldo)
    print("La base se cargó correctamente desde el archivo de respaldo. \n")

# 5. Mostramos las primeras filas
df_medicamentos.head()

La base se cargó correctamente desde PAMI.



,DROGA,MARCA,PRESENTACION,LABORATORIO,COBERTURA,COPAGO
0,aceite de pescado,OMACOR,caps.blandas x 28,Richmond,50%,$ 2483.68
1,aceite de salmón,REGULIP,1 g cáps.x 60,Adium,40%,$ 40843.77
2,aceite de salmón,REGULIP,1 g cáps.x 90,Adium,40%,$ 59464.27
3,acenocumarol,SINTROM,1 mg comp.x 20,Siegfried,80%,$ 859.88
4,acenocumarol,ACENOCOUMAROL ROSPAW,4 mg comp.x 20,Rospaw,80%,$ 2568.55


##**2. Exploración del dataset cargado**

1. Nombres de filas
2. Cantidad de filas y columnas
3. Revisar la falta de datos
4. Ver el tipo de datos, si es texto o numero

In [ ]:
# 1. Ver nombres de columnas
df_medicamentos.columns

Index(['DROGA ', 'MARCA ', 'PRESENTACION', 'LABORATORIO', 'COBERTURA',
       'COPAGO'],
      dtype='object')

In [ ]:
# 2. Cantidad de filas y columnas
df_medicamentos.shape

(8391, 6)

In [ ]:
# 3. Ver si hay valores faltantes por columna
df_medicamentos.isnull().sum()

,0
DROGA,0
MARCA,0
PRESENTACION,0
LABORATORIO,0
COBERTURA,0
COPAGO,0


In [ ]:
# 4. Tipos de datos de las columnas
df_medicamentos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8391 entries, 0 to 8390
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   DROGA         8391 non-null   object
 1   MARCA         8391 non-null   object
 2   PRESENTACION  8391 non-null   object
 3   LABORATORIO   8391 non-null   object
 4   COBERTURA     8391 non-null   object
 5   COPAGO        8391 non-null   object
dtypes: object(6)
memory usage: 393.5+ KB


## **3. Limpieza básica del dataset**

La limpieza va a permitir eliminar espacios en blanco al inicio y al final de las celdas y evitar errores por diferencias de formato, como escribir en mayusculas o minusculas.

In [ ]:
# 1. Crear copia del dataset original para tener un respaldo
df_medicamentos_limpio = df_medicamentos.copy()

# 2. Limpiar nombres de columnas para que queden en mayuscula
df_medicamentos_limpio.columns = (df_medicamentos_limpio.columns
                                  .str.strip()
                                  .str.upper())

# 3. Limpiar espacios en columnas de texto para eliminar espacion en blanco al principio y al final
for columna in df_medicamentos_limpio.select_dtypes(include="object"):
    df_medicamentos_limpio[columna] = (df_medicamentos_limpio[columna]
                                       .astype(str)
                                       .str.strip())
# 4. Como COPAGO estaba en texto, se lo convierte a numero para pdoer hacer funciones despues
df_medicamentos_limpio["COPAGO"] = (df_medicamentos_limpio["COPAGO"]
                                    .str.replace("$", "", regex=False)   # elimina el signo $
                                    .str.replace(",", "", regex=False)   # elimina comas
                                    .str.strip()                         # elimina espacios
                                    .astype(float))                       # convierte a número decimal

# 5. Cambiar nombre de la columna COPAGO porque si le quitamos el signo $ es algo dificil de interpretar para el colectivo de destinatarios
df_medicamentos_limpio = df_medicamentos_limpio.rename(columns={"COPAGO": "A_PAGAR"})

# 6. Mostrar columnas luego de la limpieza
df_medicamentos_limpio.columns

Index(['DROGA', 'MARCA', 'PRESENTACION', 'LABORATORIO', 'COBERTURA',
       'A_PAGAR'],
      dtype='object')

## **3. Buscador básico de medicamentos**

Solo lo pongo para que se vea como funciona la busqueda, le vamos a pedir que nos traiga 20 resultados

In [ ]:
# 1. Se escribe el medicamento o droga a buscar
busqueda = "metformina"

# 2. Buscamos coincidencias en las columnas DROGA y MARCA
resultado = df_medicamentos_limpio[df_medicamentos_limpio["DROGA"].astype(str).str.contains(busqueda, case=False, na=False) |
                                   df_medicamentos_limpio["MARCA"].astype(str).str.contains(busqueda, case=False, na=False)]

# 3. Mostramos los primeros 20 resultados
resultado.head(20)

,DROGA,MARCA,PRESENTACION,LABORATORIO,COBERTURA,A_PAGAR
3814,glimepiride+metformina,MECTIN PLUS,1000/2 mg comp.rec.x 30,Teva Argentina,50%,224.85
3815,glimepiride+metformina,MECTIN PLUS,1000/4 mg comp.rec.x 30,Teva Argentina,50%,268.13
3816,glimepiride+metformina,AMARYL M,2 mg/1 g comp.rec.x 30,Sanofi-Aventis,50%,361.15
3817,glimepiride+metformina,GLEMAZ MET 2/1000,2/1000 mg comp.rec.x 30,Montpellier,50%,6948.19
3818,glimepiride+metformina,ENDIAL MET 2,2/1000 mg comp.rec.x 30,Roemmers,50%,7436.50
3819,glimepiride+metformina,GLEMAZ MET 2/1000,2/1000 mg comp.rec.x 60,Montpellier,50%,13007.86
3820,glimepiride+metformina,ENDIAL MET 2,2/1000 mg comp.rec.x 60,Roemmers,50%,14040.50
3821,glimepiride+metformina,ADIUVAN MET,2/1000 mg comp.x 30,Lazar,50%,7297.32
3822,glimepiride+metformina,HIPOGLUT MET,2mg/1000mg x 30 comp.,Gador,50%,7747.59
3823,glimepiride+metformina,HIPOGLUT MET,2mg/500mg x 30 comp.,Gador,50%,7514.54


## **4. Buscador ampliado de medicamentos**

Acá se va a poder buscar más específico.

Por ejmplo, si la persona escribe “indalten amlodipina 10”, el sistema busca filas donde aparezcan esas palabras en conjunto, aunque estén no esten en la misma columna. Es decir va a buscar por fila droga, marca y presentación.

Para el usuario final, esto se vería mucho más simple “Escriba el nombre del medicamento como aparece en la caja” y ponerle un ejemplo "Por ejemplo: Indalten Amlodipina 10 mg"

Después el codigo mostraria mostrar pocas opciones pediria que “Seleccione el medicamento correcto”

Esto simplifica el uso porque el usuario no tiene que saber la diferencia entre droga, marca, laboratorio o presentación. Esto es importante porque el grupo al cual se esta dirigiendo el sitio pueden tener dificultades de interpretación

In [ ]:
busqueda = "indalten amlodipina 10"

# 1. Se convierte la búsqueda en palabras separadas
palabras = busqueda.lower().split()

# 2. Columnas donde se quiere buscar
columnas_busqueda = ["DROGA", "MARCA", "PRESENTACION", "LABORATORIO"]

# 3. Se crea  una columna auxiliar que junta la información útil de cada medicamento
df_medicamentos_limpio["TEXTO_BUSQUEDA"] = (df_medicamentos_limpio[columnas_busqueda]
                                            .astype(str)
                                            .agg(" ".join, axis=1)
                                            .str.lower())

# 4. Se filtran filas que contengan todas las palabras buscadas
resultado = df_medicamentos_limpio[df_medicamentos_limpio["TEXTO_BUSQUEDA"]
                                   .apply(lambda texto: all(palabra in texto for palabra in palabras))]

# 5. Se muestran solo columnas útiles
resultado[["DROGA", "MARCA", "PRESENTACION", "LABORATORIO", "COBERTURA", "A_PAGAR"]].head(20)

,DROGA,MARCA,PRESENTACION,LABORATORIO,COBERTURA,A_PAGAR
292,amlodipina,INDALTEN,10 mg comp.x 30,Montpellier,60%,7068.05


## **5. Creación de una función reutilizable de búsqueda**
El buscador de medicamentos, como ya funciona, lo volvemos "reutilizable" para que el usuario pueda cargar todos los emdicamentos que quiera

---
**NOTA PARA ENTENDER EL CODIGO**

* En la celda de codigo siguiente se va a poder entender porque cada seccion va a tener el numero correspondiente a la explicación siguiente*

1. La función recibe dos elementos:

    * `busqueda` → el texto que escribe el usuario.
    * `df` → el dataset donde se realizará la búsqueda.

2. La búsqueda se divide en palabras individuales, cada palabra se separa con una coma, para permitir búsquedas más flexibles.
    * Por ejemplo si ingresan "indalten amlodipina 10" se convierte en "indalten", "amlodipina", "10"

3. La búsqueda se realizará sobre varias columnas del dataset para encontrar medicamentos aunque el usuario escriba solamente parte de la información de la caja.
    * Solo vamos a poner las columnas que refieren al medicamente, no utilizando por ahora las columnas de "cobertura" y "a_pagar"

4. Se crea una copia temporal para evitar modificar accidentalmente el dataset original durante la búsqueda.

5. Se crea la columna auxiliar `TEXTO_BUSQUEDA` que junta toda la información importante del medicamento en un único texto.

6. El sistema revisa qué filas contienen todas las palabras escritas por el usuario.

7. La columna auxiliar se elimina antes de mostrar el resultado final para mantener el dataset limpio.

In [ ]:
def buscar_medicamento(busqueda, df): # 1

  palabras = busqueda.lower().split() # 2

  columnas_busqueda = ["DROGA","MARCA","PRESENTACION","LABORATORIO"] # 3

  df = df.copy() # 4

  df["TEXTO_BUSQUEDA"] = (df[columnas_busqueda]
                          .astype(str)
                          .agg(" ".join, axis=1)
                          .str.lower()) # 5

  resultado = df[df["TEXTO_BUSQUEDA"].apply(lambda texto: all(palabra in texto for palabra in palabras))] # 6

  return resultado.drop(columns=["TEXTO_BUSQUEDA"]) # 7


In [ ]:
# PRUEBA DE LA FUNCIÓN
# =========================

buscar_medicamento("indalten amlodipina 10",df_medicamentos_limpio)

,DROGA,MARCA,PRESENTACION,LABORATORIO,COBERTURA,A_PAGAR
292,amlodipina,INDALTEN,10 mg comp.x 30,Montpellier,60%,7068.05


# **6. Lista de medicamentos seleccionados**

Se crea una lista vacia donde posteriormente se van a guardar los medicamentos que el usuario seleccione.
Se coloca acá, y no despues, simplemente porque me puteo el código

In [ ]:
medicamentos_seleccionados = []

## **7. Selección de medicamentos encontrados**

Con un `input()` se va a simular el funcionamiento futuro de la página web.Vamos a hacer de cuenta que el usuario escribe el nombre del medicamento como aparece en la caja y vamos a ir guardando los resultados seleccionados en la lista creada en el punto 6

---
**NOTA**

En google colab no podemos hacer que la gente clicke la opcion que quiere, entonces solo para ver que funciona y como prueba, cuando aparezcan varias opciones, la persona pueda elegir cuál corresponde a su caja o receta.

Luego, el medicamento elegido se guarda en una lista temporal para poder calcular más adelante el total a pagar.

In [ ]:
# 1. Antes del input se va a dar un mensaje lo más simple y claro posible para que el grupo de destino lo comprenda facilmente
print("Para buscar, escriba el nombre del medicamento como aparece en la caja. \n \nEs importante que si el nombre cuenta con acento se lo coloque. \n \nCuando termine de colocar todos los medicamentos escribra 'SALIR' \n")

# 2. Generamos un bucle dejando por fuera el mensaje anterior y repitiendo el input las veces que sea necesario
while True:

    # 3. Pedir al usuario el medicamento, quitamos espacios de mas al inicio y final y ponemos todo en mayuscula para que admita el texto
    medicamento_usuario = input("\n Nombre del medicamento: \n").strip().lower()

    # 4. Si escribe FIN se termina el bucle, como la estandarizacion de formato la pusimos en el paso anterior no es necesaria
    if medicamento_usuario == "salir":
        print("\nBúsqueda finalizada. \n")
        break

    # 5. Ejecutar búsqueda usando la funcion reutilizable
    resultado_usuario = buscar_medicamento(medicamento_usuario,df_medicamentos_limpio)

    # 6. Mostrar un mensaje si no hay opciones que coincidan
    if resultado_usuario.empty:
        print("\nNo encontramos medicamentos con ese nombre.")
        print("Revise si está bien escrito. También puede probar escribiendo solo el nombre principal del medicamento. \n")

    # 7. Mostrar resultados todos los resultados que coinciden con la busqueda
      # Se prevee que la persona no coloque la cantidad de comprimidos o los mg o ml del medicamento, incluso que no coloque el laboratorio

    # a. Vamos a reiniciar el indice, no vamos a poner el numero del dataset sino desde 1 para que la persona pueda seleccionar la opción facilmente
    else:

        opciones = resultado_usuario.reset_index(drop=True)
        opciones.index = opciones.index + 1

        # solo mostramos columnas que permiten al usurio encontrar el medicamento adecuado
        print("\n Estas son las opciones encontradas: \n")
        display(opciones[["DROGA","MARCA","PRESENTACION","LABORATORIO"]])

        # vamos a añadir una fila al resultado, asi el usuario no se ve obligado a elegir un medicamento si el suyo no esta
        opcion_ninguna = len(opciones) + 1

        print(f"\n{opcion_ninguna}. Ninguna opción corresponde a mi medicamento.")

        # Si el numero esta lo obligamos a elegir la opcion que corresponde
        seleccion = input("\nEscriba el número de la opción correcta: \n").strip()

        if seleccion.isdigit():

            seleccion = int(seleccion)

            # Si el usuario no encontro su medicamento lo mandamos al input inicial
            if seleccion == opcion_ninguna:
                print("\nNo se agregó ningún medicamento. Puede buscar nuevamente.")

            # Si el usuario eligio una opcion entonces guardamos el medicamento seleccionado en la lista que hasta ahora teniamos vacia
            elif seleccion in opciones.index:

                medicamento_elegido = opciones.loc[seleccion]

                medicamentos_seleccionados.append(medicamento_elegido)

                print("\n Medicamento agregado correctamente: \n")

                display(medicamento_elegido[["DROGA","MARCA","PRESENTACION","LABORATORIO"]])

            # Le dejamos mensajes si no ponen una opcion correcta
            else:
                print("\nEl número ingresado no corresponde a ninguna opción.")

        else:
            print("\nDebe escribir solamente el número de la opción.")

Para buscar, escriba el nombre del medicamento como aparece en la caja. 
 
Es importante que si el nombre cuenta con acento se lo coloque. 
 
Cuando termine de colocar todos los medicamentos escribra 'SALIR' 


 Nombre del medicamento: 
nuevapina cardio

 Estas son las opciones encontradas: 



,DROGA,MARCA,PRESENTACION,LABORATORIO
1,"acetilsalicílico,ác.",NUEVAPINA CARDIO,100 mg comp.x 50,Savant Generic



2. Ninguna opción corresponde a mi medicamento.

Escriba el número de la opción correcta: 
1

 Medicamento agregado correctamente: 



,1
DROGA,"acetilsalicílico,ác."
MARCA,NUEVAPINA CARDIO
PRESENTACION,100 mg comp.x 50
LABORATORIO,Savant Generic



 Nombre del medicamento: 
sincerum

 Estas son las opciones encontradas: 



,DROGA,MARCA,PRESENTACION,LABORATORIO
1,"antipirina+sodio,carbonato+asoc.",SINCERUM,gts.x 5 ml,Dallas



2. Ninguna opción corresponde a mi medicamento.

Escriba el número de la opción correcta: 
1

 Medicamento agregado correctamente: 



,1
DROGA,"antipirina+sodio,carbonato+asoc."
MARCA,SINCERUM
PRESENTACION,gts.x 5 ml
LABORATORIO,Dallas



 Nombre del medicamento: 
salir

Búsqueda finalizada. 



## **8. Visualización de la lista de medicamentos seleccionados**

Vamos a guardar la lista como si fuera un base de datos nueva.

Cada vez que reiniciemos el espacio la lista vuelve a estar vacia asi que no hay una acumulación de datos

y lo ponemos con la función display para que quede visualmente lindo


In [ ]:

# La lista que tenemos con los emdicamento la convertimos en un DataFrame nuevo
df_seleccionados = pd.DataFrame(medicamentos_seleccionados)

# Mostramos la tabla
display(df_seleccionados)

# Sumamos los valores de todos los medicamentos
total_a_pagar = df_seleccionados["A_PAGAR"].sum()

# Vamos a mostrar el total, y le ponemos el signo pesos porque se lo habiamos quitado cuando lo pasamos a numero
print(f"\n Total estimado a pagar: ${total_a_pagar:}")

,DROGA,MARCA,PRESENTACION,LABORATORIO,COBERTURA,A_PAGAR
1,"acetilsalicílico,ác.",NUEVAPINA CARDIO,100 mg comp.x 50,Savant Generic,60%,2837.96
1,"antipirina+sodio,carbonato+asoc.",SINCERUM,gts.x 5 ml,Dallas,50%,4008.22



 Total estimado a pagar: $6846.18
